# Airfare Route Segmentation by Historical Pricing Behaviour
## Phase 1: Data Source Preparation, Feature Engineering & Scaling
**Course / Domain**: Unsupervised Machine Learning & Revenue Management  
**Dataset**: Kaggle 'Flight Price Prediction' Dataset (`Clean_Dataset.csv`)  
**Objective**: Prepare raw flight observations, handle missing values, encode categorical variables, aggregate route-level pricing behavior, and scale numerical matrices for clustering.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# Add src to path
sys.path.append(os.path.abspath("../src"))
from data_loader import load_or_generate_dataset
from feature_engineering import preprocess_flight_records, aggregate_route_features, prepare_clustering_matrix

print("Environment configured successfully.")


### 1. Dataset Loading and Schema Verification
We inspect the Kaggle Flight Price Prediction dataset schema containing over 50,000 observations across major domestic and international commercial flight corridors.


In [ ]:
df = load_or_generate_dataset(num_samples=50000)
print(f"Dataset Shape: {df.shape}")
df.head(10)


### 2. Handling Missing Values and Categorical Encoding
- Missing values in numerical attributes are imputed with the median.
- Missing values in categorical attributes are imputed with the mode.
- Ordinal encoding is applied to `stops` (zero: 0, one: 1, two_or_more: 2) and `class` (Economy: 0, Business: 1).


In [ ]:
clean_df = preprocess_flight_records(df)
print("Missing values summary after imputation:")
print(clean_df.isnull().sum())
print("
Transformed columns:")
print(clean_df[['route', 'stops_num', 'class_num', 'is_last_minute', 'is_peak_hour']].head())


### 3. Route-Level Behavioral Aggregation
In revenue management, pricing behavior is characterized at the **route corridor level** by calculating:
- Mean and Median Fare (central pricing scale)
- Price Volatility (Standard Deviation & Coefficient of Variation $CV = \sigma / \mu$)
- Yield Management Aggressiveness (Last-Minute Surge Ratio: fare at $\le$ 3 days vs $\ge$ 21 days)
- Price Spread (90th minus 10th percentile)
- Average Duration and Direct Flight Ratio
- Business Class Demand Share


In [ ]:
route_df = aggregate_route_features(clean_df)
print(f"Aggregated {len(route_df)} commercial route corridors.")
route_df.head()


### 4. Numerical Standardization (Feature Scaling)
Because distance-based algorithms such as K-Means are sensitive to variable magnitudes (e.g. Fares in thousands vs Duration in single digits), we apply `StandardScaler`:
$$z = \frac{x - \mu}{\sigma}$$


In [ ]:
X_scaled, scaler, feature_names = prepare_clustering_matrix(route_df)
print("Scaled matrix shape:", X_scaled.shape)
print("Features used for clustering:")
for idx, f in enumerate(feature_names):
    print(f"  {idx+1}. {f} (mean={X_scaled[:, idx].mean():.2f}, std={X_scaled[:, idx].std():.2f})")
